In [2]:
!pip install nflreadpy pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nflreadpy]


In [3]:
import numpy as np
import pandas as pd
import nflreadpy as nfl

CACHE = "pbp_2015_2025.parquet"
try:
    pbp = pd.read_parquet(CACHE)
except FileNotFoundError:
    pbp = nfl.load_pbp(list(range(2015, 2026))).to_pandas()
    pbp.to_parquet(CACHE)

print(pbp.shape)

(532376, 372)


In [4]:
cols = ["season", "home_team", "posteam", "score_differential",
        "game_seconds_remaining", "yardline_100", "result"]
df = pbp[cols].dropna()        # drop plays with missing values (kickoffs, timeouts, games not played yet)
df = df[df["result"] != 0]     # drop tie games
print(df.shape)

(492773, 7)


In [5]:
df["home_win"] = (df["result"] > 0).astype(int)
print(df["home_win"].mean())

0.5488713870281042


In [6]:
home_has_ball = (df["posteam"] == df["home_team"]).astype(int)

# score_differential = (team with ball) − (other team). Flip it when the away team has the ball.
df["home_lead"] = np.where(home_has_ball == 1, df["score_differential"], -df["score_differential"])

# yardline_100 = yards the team with the ball needs to score.
# Convert it to: how far the ball is from the end zone the home team attacks.
df["home_yards_to_score"] = np.where(home_has_ball == 1, df["yardline_100"], 100 - df["yardline_100"])

df["home_has_ball"] = home_has_ball
df["time_left"] = df["game_seconds_remaining"]

features = ["home_lead", "time_left", "home_has_ball", "home_yards_to_score"]
print(df[features + ["home_win"]].head(10))

    home_lead  time_left  home_has_ball  home_yards_to_score  home_win
1        -0.0     3600.0              0                 65.0         1
2        -0.0     3600.0              0                 20.0         1
3        -0.0     3573.0              0                 23.0         1
4        -0.0     3532.0              0                 25.0         1
5        -0.0     3494.0              0                 35.0         1
6        -0.0     3450.0              0                 35.0         1
7        -0.0     3405.0              0                 38.0         1
8        -0.0     3401.0              0                 38.0         1
9         0.0     3393.0              1                 86.0         1
10        0.0     3372.0              1                 83.0         1


In [7]:
train = df[df["season"] <= 2022]
test  = df[df["season"] >= 2023]

X_train = train[features].values.astype(float)
y_train = train["home_win"].values
X_test  = test[features].values.astype(float)
y_test  = test["home_win"].values

mu, sd = X_train.mean(axis=0), X_train.std(axis=0)   # use ONLY the training data for these
X_train_s = (X_train - mu) / sd
X_test_s  = (X_test - mu) / sd

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)   # keeps log(0) from crashing
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

In [9]:
n, d = X_train_s.shape
w = np.zeros(d)
b = 0.0
lr = 0.1

for epoch in range(2000):
    p = sigmoid(X_train_s @ w + b)       # 1. predict
    error = p - y_train                  # 2. guess minus truth
    grad_w = X_train_s.T @ error / n     # 3. gradients
    grad_b = error.mean()
    w -= lr * grad_w                     # 4. step downhill
    b -= lr * grad_b
    if epoch % 200 == 0:
        print(epoch, round(log_loss(y_train, p), 4))

print("w:", dict(zip(features, w.round(3))), " b:", round(b, 3))

0 0.6931
200 0.4987
400 0.495
600 0.4947
800 0.4947
1000 0.4947
1200 0.4947
1400 0.4947
1600 0.4947
1800 0.4947
w: {'home_lead': np.float64(1.884), 'time_left': np.float64(0.029), 'home_has_ball': np.float64(0.253), 'home_yards_to_score': np.float64(-0.181)}  b: 0.303


In [10]:
from sklearn.linear_model import LogisticRegression

sk = LogisticRegression(penalty=None, max_iter=1000).fit(X_train_s, y_train)
print("mine:   ", np.round(np.r_[b, w], 3))
print("sklearn:", np.round(np.r_[sk.intercept_, sk.coef_[0]], 3))

mine:    [ 0.303  1.884  0.029  0.253 -0.181]
sklearn: [ 0.303  1.884  0.029  0.253 -0.181]


In [11]:
p_test = sigmoid(X_test_s @ w + b)
baseline = np.full_like(p_test, y_train.mean(), dtype=float)   # always guess about 55%

print("Baseline log loss:", round(log_loss(y_test, baseline), 4))
print("My model log loss:", round(log_loss(y_test, p_test), 4))
print("Accuracy:", round(((p_test >= 0.5) == y_test).mean(), 3))

Baseline log loss: 0.6885
My model log loss: 0.4953
Accuracy: 0.744


In [12]:
def win_prob(lead, secs_left, has_ball, yards_to_score):
    x = (np.array([lead, secs_left, has_ball, yards_to_score]) - mu) / sd
    return sigmoid(x @ w + b)

print("Up 7, 2 min left, home ball at midfield:", round(win_prob(7, 120, 1, 50), 3))
print("Tied at kickoff:                        ", round(win_prob(0, 3600, 0, 75), 3))
print("Down 7, 2 min left, away ball:          ", round(win_prob(-7, 120, 0, 50), 3))

Up 7, 2 min left, home ball at midfield: 0.824
Tied at kickoff:                         0.427
Down 7, 2 min left, away ball:           0.191
